In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
spark = SparkSession.builder.appName("Read Parquet").getOrCreate()
# read in all the data across the times
year_2026_01 = spark.read.parquet("data/year-2026/yellow_tripdata_2026-01.parquet")
year_2026_02 = spark.read.parquet("data/year-2026/yellow_tripdata_2026-02.parquet")
year_2026_03 = spark.read.parquet("data/year-2026/yellow_tripdata_2026-03.parquet")
year_2026_04 = spark.read.parquet("data/year-2026/yellow_tripdata_2026-04.parquet")
year_2026_05 = spark.read.parquet("data/year-2026/yellow_tripdata_2026-05.parquet")
data = year_2026_01.union(year_2026_02).union(year_2026_03).union(year_2026_04).union(year_2026_05)
data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
print(data.count())
data.select('tpep_pickup_datetime').distinct().count()

18999282


8781561

# Calculating trip duration and revenue per minute

In [ ]:
data1 = data
data1 = data1.withColumn('duration_minute', F.timestamp_diff('MINUTE', 'tpep_pickup_datetime', 'tpep_dropoff_datetime'))
zone_stats = data1.groupby('PULocationID').agg({'total_amount': 'sum', 'duration_minute': 'sum'}).withColumnsRenamed({'sum(duration_minute)': 'total_min', 'sum(total_amount)': 'total_revenue'}) 
zone_stats = zone_stats.withColumn('revenue_per_min', F.col('total_revenue')/F.col('total_min'))
zone_stats.show()

+------------+------------------+---------+------------------+------------------+
|PULocationID|     total_revenue|total_min|   revenue_per_min|   revenue_per_day|
+------------+------------------+---------+------------------+------------------+
|         148|  7173216.48999992|  3850676|1.8628460275546217|2682.4982796786553|
|         243| 796995.7299999984|   524808|1.5186424940168564| 2186.845191384273|
|          31|24943.809999999998|    18552| 1.344534821043553|1936.1301423027166|
|         137|5912935.3600000935|  3625831|1.6307807396428828|2348.3242650857514|
|          85|         278465.54|   278239|1.0008141921154114|1441.1724366461924|
|         251|           6042.42|     2788|2.1672955523672885|3120.9055954088954|
|          65| 879406.6699999995|   599500|1.4669002001668048| 2112.336288240199|
|         255| 1637274.800000008|   927572|1.7651188263552673| 2541.771109951585|
|          53| 68854.93000000002|    50923|1.3521381301180218|1947.0789073699516|
|         133|18

In [7]:
import geopandas as gpd
taxi_zones = gpd.read_file('data/taxi_zones/taxi_zones.shp')
print(taxi_zones.dtypes)
taxi_zones.head()

OBJECTID         int32
Shape_Leng     float64
Shape_Area     float64
zone               str
LocationID       int32
borough            str
geometry      geometry
dtype: object


,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, 992068.667 20..."
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, 936046.565 144..."


In [8]:
neighbours = gpd.sjoin(
    taxi_zones[['OBJECTID', 'geometry']],
    taxi_zones[['OBJECTID', 'geometry']],
    how='left',
    predicate='touches'
)
neighbours_id = (
    neighbours
    .groupby('OBJECTID_left')['OBJECTID_right']
    .agg(list)
    .reset_index()
    .rename(columns={'OBJECTID_left': 'ID', 'OBJECTID_right': 'neighbour_ID'})
)
print(neighbours_id.dtypes)
neighbours_id

ID               int32
neighbour_ID    object
dtype: object


,ID,neighbour_ID
0,1,[nan]
1,2,[132.0]
2,3,"[242.0, 184.0, 51.0, 254.0]"
3,4,"[148.0, 79.0, 224.0]"
4,5,[nan]
...,...,...
258,259,[254.0]
259,260,[157.0]
260,261,"[88.0, 209.0, 231.0]"
261,262,"[140.0, 141.0, 263.0, 75.0]"


In [9]:
import math
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    ArrayType
)
from pyspark.sql import functions as F


# ============================================================
# 1. Convert GeoPandas neighbour data into Python records
# ============================================================

records = []

for _, row in neighbours_id.iterrows():

    neighbour_list = row["neighbour_ID"]

    # Remove NaN values from the neighbour list
    if isinstance(neighbour_list, list):
        cleaned_neighbours = [
            int(x)
            for x in neighbour_list
            if x is not None
            and not (isinstance(x, float) and math.isnan(x))
        ]
    else:
        cleaned_neighbours = []

    records.append(
        (
            int(row["ID"]),
            cleaned_neighbours
        )
    )


# ============================================================
# 2. Create Spark DataFrame containing the neighbours
# ============================================================

neighbour_schema = StructType([
    StructField(
        "ID",
        IntegerType(),
        nullable=False
    ),
    StructField(
        "neighbour_ID",
        ArrayType(IntegerType()),
        nullable=True
    )
])

neighbours_spark = spark.createDataFrame(
    records,
    schema=neighbour_schema
)


# ============================================================
# 3. Turn each neighbour in the list into its own row
# ============================================================

neighbours_exploded = neighbours_spark.select(
    "ID",
    F.explode("neighbour_ID").alias("neighbour_ID")
)


# ============================================================
# 4. Get revenue_per_min for each neighbour
# ============================================================

neighbour_revenue = neighbours_exploded.join(
    zone_stats.select(
        F.col("PULocationID"),
        F.col("revenue_per_min").alias(
            "neighbour_revenue_per_min"
        )
    ),
    neighbours_exploded["neighbour_ID"]
        == F.col("PULocationID"),
    how="left"
)


# ============================================================
# 5. Calculate average revenue_per_min across neighbours
# ============================================================

expected_revenue = neighbour_revenue.groupBy(
    "ID"
).agg(
    F.avg(
        "neighbour_revenue_per_min"
    ).alias(
        "expected_revenue_per_min"
    )
)


# ============================================================
# 6. Add expected revenue back onto zone_stats
# ============================================================

zone_stats = zone_stats.join(
    expected_revenue,
    zone_stats["PULocationID"]
        == expected_revenue["ID"],
    how="left"
).drop(
    expected_revenue["ID"]
)


# ============================================================
# 7. Check the result
# ============================================================

zone_stats.select(
    "PULocationID",
    "revenue_per_min",
    "expected_revenue_per_min",
).show()

+------------+------------------+------------------------+
|PULocationID|   revenue_per_min|expected_revenue_per_min|
+------------+------------------+------------------------+
|         148|1.8628460275546217|      1.7194977396183613|
|         243|1.5186424940168564|                    NULL|
|          31| 1.344534821043553|      0.9403038098092695|
|         137|1.6307807396428828|      1.7571796665882706|
|          85|1.0008141921154114|      0.9850897797601564|
|         251|2.1672955523672885|       2.123026316640638|
|          65|1.4669002001668048|      1.3531411411908931|
|         255|1.7651188263552673|      1.6643003281063158|
|          53|1.3521381301180218|       2.239888663316662|
|         133|1.2087946676420454|      0.9319665062873244|
|          78|0.9158101016515692|       0.947592135434454|
|         108| 0.817569792652307|      0.8265202127016739|
|         155|0.9052566751283144|      0.8411457014364051|
|         211|1.7445949314963234|      1.777800793415959

In [10]:
zone_stats = zone_stats.withColumn(
    "underperformance_pct",
    ((F.col("expected_revenue_per_min") - F.col("revenue_per_min")) 
     / F.col("expected_revenue_per_min") * 100)
)

zone_stats = zone_stats.withColumn(
    "is_underperformer",
    F.col("underperformance_pct") > 10  # Flag zones >10% below neighbors
)

In [19]:
city_avg = zone_stats.agg(
    F.avg("revenue_per_min").alias("city_avg_revenue")
).collect()[0][0]

zone_stats = zone_stats.withColumn(
    "city_avg_revenue",
    F.lit(city_avg)
).withColumn(
    "below_city_avg_pct",
    ((F.col("city_avg_revenue") - F.col("revenue_per_min")) 
     / F.col("city_avg_revenue") * 100)
)
print(zone_stats.count())
zone_stats.show()

262


+------------+------------------+---------+------------------+------------------+------------------------+--------------------+-----------------+------------------+-------------------+
|PULocationID|     total_revenue|total_min|   revenue_per_min|   revenue_per_day|expected_revenue_per_min|underperformance_pct|is_underperformer|  city_avg_revenue| below_city_avg_pct|
+------------+------------------+---------+------------------+------------------+------------------------+--------------------+-----------------+------------------+-------------------+
|         148|  7173216.48999992|  3850676|1.8628460275546217|2682.4982796786553|      1.7194977396183613|  -8.336637183836965|            false|1.8825962772551676| 1.0490963962460336|
|         243| 796995.7299999984|   524808|1.5186424940168564| 2186.845191384273|                    NULL|                NULL|             NULL|1.8825962772551676| 19.332545572062706|
|          31|24943.809999999998|    18552| 1.344534821043553|1936.13014230

In [12]:
raw_weather_data = spark.read.csv("data/weather.csv", header=True, inferSchema=True)
print(raw_weather_data.columns)

['STATION', 'NAME', 'DATE', 'AWND', 'DAPR', 'MDPR', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'TOBS', 'WDF2', 'WDF5', 'WESD', 'WESF', 'WSF2', 'WSF5', 'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT08', 'WT09', 'WT11']


In [13]:
raw_weather_data = raw_weather_data.filter(
    F.col('STATION').isin(['USW00014732', 'USW00094789'])  # LaGuardia + JFK
)

In [14]:
weather_df = raw_weather_data.filter(raw_weather_data['NAME'].contains('LAGUARDIA')).select('DATE', 'PRCP', 'SNOW', 'TMAX', 'TMIN')
weather_df = weather_df.withColumn('TAVG', (F.col('TMAX') + F.col('TMIN')) / 2)
weather_df.show()

+----------+----+----+----+----+----+
|      DATE|PRCP|SNOW|TMAX|TMIN|TAVG|
+----------+----+----+----+----+----+
|2026-01-01| 0.0| 0.0|  36|  23|29.5|
|2026-01-02| 0.0| 0.0|  30|  20|25.0|
|2026-01-03| 0.0| 0.0|  31|  24|27.5|
|2026-01-04| 0.0| 0.0|  35|  26|30.5|
|2026-01-05| 0.0| 0.0|  38|  30|34.0|
|2026-01-06|0.01| 0.0|  39|  35|37.0|
|2026-01-07|0.01| 0.0|  48|  36|42.0|
|2026-01-08| 0.0| 0.0|  50|  37|43.5|
|2026-01-09| 0.0| 0.0|  54|  35|44.5|
|2026-01-10|0.46| 0.0|  53|  38|45.5|
|2026-01-11|0.02| 0.0|  47|  34|40.5|
|2026-01-12| 0.0| 0.0|  41|  32|36.5|
|2026-01-13| 0.0| 0.0|  47|  37|42.0|
|2026-01-14| 0.0| 0.0|  51|  42|46.5|
|2026-01-15| 0.0| 0.0|  46|  25|35.5|
|2026-01-16| 0.0| 0.0|  34|  22|28.0|
|2026-01-17|0.17| 1.4|  40|  31|35.5|
|2026-01-18|0.26| 1.5|  35|  31|33.0|
|2026-01-19| 0.0| 0.0|  33|  24|28.5|
|2026-01-20| 0.0| 0.0|  27|  17|22.0|
+----------+----+----+----+----+----+
only showing top 20 rows


In [15]:
# Get your underperformers from Phase 3
underperformers = zone_stats.filter(
    (F.col('underperformance_pct') > 8) &
    (F.col('below_city_avg_pct') > 10)
).select('PULocationID').collect()

underperformer_ids = [row.PULocationID for row in underperformers]

# Now filter taxi data to only these zones
taxi_underperformers = data1.filter(
    F.col('PULocationID').isin(underperformer_ids)
)
taxi_underperformers.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|duration_minute|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+
|       2| 2026-01-01 00:42:34|  2026-01-01 00:47:15|              1|         0.55

In [16]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

w = Window.orderBy("date")

# choose desired zone to visualise
desired_underperformer = taxi_underperformers.filter(F.col("PULocationID") == 140)
# aggregate to a daily frequency
daily_revenue = desired_underperformer.groupBy(
    F.to_date('tpep_pickup_datetime').alias('date'),
    'PULocationID'
).agg(
    F.sum('total_amount').alias('daily_revenue'),
    F.count('*').alias('trip_count'),
    F.avg('total_amount').alias('avg_trip_revenue')
).orderBy('date')
daily_revenue.show()

+----------+------------+------------------+----------+------------------+
|      date|PULocationID|     daily_revenue|trip_count|  avg_trip_revenue|
+----------+------------+------------------+----------+------------------+
|2026-01-01|         140|32481.030000000035|      1393|23.317322325915317|
|2026-01-02|         140| 43681.00999999996|      1924| 22.70322765072763|
|2026-01-03|         140|32519.280000000028|      1482|21.942834008097186|
|2026-01-04|         140|27393.150000000023|      1279|21.417630961688836|
|2026-01-05|         140| 64206.22999999997|      2751| 23.33923300617956|
|2026-01-06|         140| 72347.66999999993|      3039| 23.80640671273443|
|2026-01-07|         140| 73551.28999999998|      3025|24.314476033057844|
|2026-01-08|         140| 74526.60999999996|      2999|24.850486828942966|
|2026-01-09|         140| 70137.26000000005|      2922|24.003169062286123|
|2026-01-10|         140| 49360.88999999997|      2164| 22.81002310536043|
|2026-01-11|         140|

In [17]:
taxi_underperformers_sub = daily_revenue.withColumn("row_num", F.row_number().over(w))
taxi_underperformers_sub = taxi_underperformers_sub.filter((F.col("row_num") % 2) == 1).drop("row_num")

In [18]:
import pandas as pd
expected_val = zone_stats.filter(zone_stats.PULocationID == 140).select('expected_revenue_per_day').collect()[0][0]
# Convert PySpark DataFrame to Pandas for plotting
taxi_underperformers_sub = taxi_underperformers_sub.select("date", "daily_revenue") \
    .orderBy("date") \
    .toPandas()

# Plot
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(taxi_underperformers_sub["date"], taxi_underperformers_sub["daily_revenue"])
# Plot constant line
plt.plot(
    taxi_underperformers_sub["date"],
    [expected_val] * len(taxi_underperformers_sub),
    label="Expected Revenue per Min (Zone 140)"
)
plt.xlim(pd.Timestamp('2026-01-01'), pd.Timestamp('2026-05-31'))
plt.xlabel("Date")
plt.ylabel("Daily Revenue")
plt.title("Total Amount over Time for Taxi Underperformers")
plt.tight_layout()
plt.show()

{"ts": "2026-08-12 16:14:03.760", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `expected_revenue_per_day` cannot be resolved. Did you mean one of the following? [`expected_revenue_per_min`, `revenue_per_day`, `revenue_per_min`, `total_revenue`, `city_avg_revenue`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "104)", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o398.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `expected_revenue_per_day` cannot be resolved. Did you mean one of the following? [`expected_revenue_per_min`, `revenue_per_day`, `revenue_per_min`, `total_revenue`, `city_avg

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `expected_revenue_per_day` cannot be resolved. Did you mean one of the following? [`expected_revenue_per_min`, `revenue_per_day`, `revenue_per_min`, `total_revenue`, `city_avg_revenue`]. SQLSTATE: 42703;
'Project ['expected_revenue_per_day]
+- Filter (PULocationID#136 = 140)
   +- Project [PULocationID#136, total_revenue#285, total_min#284L, revenue_per_min#286, revenue_per_day#287, expected_revenue_per_min#321, underperformance_pct#486, is_underperformer#487, city_avg_revenue#524, (((city_avg_revenue#524 - revenue_per_min#286) / city_avg_revenue#524) * cast(100 as double)) AS below_city_avg_pct#525]
      +- Project [PULocationID#136, total_revenue#285, total_min#284L, revenue_per_min#286, revenue_per_day#287, expected_revenue_per_min#321, underperformance_pct#486, is_underperformer#487, 1.8825962772551676 AS city_avg_revenue#524]
         +- Project [PULocationID#136, total_reve

In [ ]:
print(zone_stats.filter(zone_stats.PULocationID == 140).select('expected_revenue_per_day').collect()[0][0])

2626.864791833104
